# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiaburdibaloch-ui/flyrank-rafia/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

For my Refresh / Content Opportunity Scoring lane, the unit of analysis is **one content item for one client on one report date** in the `fact_content_daily_performance` table.

For this assignment, I will start with the **March 2026** monthly partition (`month=2026-03`). This gives me a mid-panel month for development rather than using the final June 2026 `_sample` month.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')"
)

print("Connected to Hugging Face warehouse.")

Connected to Hugging Face warehouse.


In [27]:
rel = "hf://datasets/FlyRank/internship-warehouse"

schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""")

schema

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `scroll_events`. These are candidate signals describing observed search performance and page engagement.

**Label / proxy:** For this development stage, I will use a defined opportunity proxy rather than claim that I have a true future outcome label. The proxy will be clearly defined from observed data and treated as decision-support only.

**Context:** `content_hash_id`, `client_hash_id`, and `report_date`. These identify the content, client, and observation date and describe the unit of analysis.

**Excluded:** Future performance information that would only be known after the decision moment. I will exclude future outcome information from the features to avoid leakage.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### What the three checks show

**Grain:** March contains 9,841,378 rows and 9,841,378 unique combinations of `client_hash_id`, `content_hash_id`, and `report_date`, with 0 duplicate grain rows. This supports the contract that one row represents one client, one content item, and one report date.

**Time window:** The March 2026 partition contains 9,841,378 rows, covering 2026-03-01 through 2026-03-31.

**Availability:** 3,611,061 rows have `gsc_data_available IS TRUE`. I will use this availability condition when working with GSC-based features so that unavailable GSC data is not treated as observed performance.

### Five candidate features

I will start with five candidate features from the March 2026 warehouse slice:

1. **`gsc_impressions`** — search visibility; knowable at the decision moment because it is already observed in the available reporting data.
2. **`gsc_clicks`** — search clicks; knowable at the decision moment because these clicks have already been recorded.
3. **`gsc_avg_position`** — average search position; knowable at the decision moment because it is calculated from observed Search Console performance.
4. **`ga4_sessions`** — Analytics sessions; knowable at the decision moment when GA4 data is available for the row.
5. **`scroll_events`** — observed page engagement; knowable at the decision moment when the corresponding Analytics data is available.

These are candidate decision-support features, not proof that a page should be refreshed. Availability will be checked before interpreting a signal.

In [29]:
features = con.sql("""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10
""").df()

features

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>
5,239,1,7.347280,<NA>,<NA>
6,191,0,7.832461,<NA>,<NA>
7,55,0,3.272727,<NA>,<NA>
8,77,0,5.636364,<NA>,<NA>
9,2,0,4.500000,<NA>,<NA>


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_grain_rows,
    COUNT(*) - COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS duplicate_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

q1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┬──────────────────────┐
│ total_rows │ unique_grain_rows │ duplicate_grain_rows │
│   int64    │       int64       │        int64         │
├────────────┼───────────────────┼──────────────────────┤
│    9841378 │           9841378 │                    0 │
└────────────┴───────────────────┴──────────────────────┘

In [31]:
q2 = con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

q2

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [32]:
q3 = con.sql("""
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""")

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ gsc_available_rows │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘

### Deliberate leakage check

To demonstrate leakage, I will create a simple proxy label from the observed March data and then deliberately create a feature directly from that label. If the feature gives a perfect match, that is not real predictive performance because the feature contains the answer. I will remove the leaked feature before continuing.

In [33]:
leak_df = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10000
""").df()

leak_df["opportunity_proxy"] = (
    leak_df["gsc_impressions"] > leak_df["gsc_impressions"].median()
).astype(int)

leak_df.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,opportunity_proxy
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,1
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,0


In [34]:
leak_df["leaked_feature"] = leak_df["opportunity_proxy"]

agreement = (
    leak_df["leaked_feature"] == leak_df["opportunity_proxy"]
).mean()

print("Agreement between leaked feature and proxy:", agreement)

Agreement between leaked feature and proxy: 1.0


In [35]:
leak_df = leak_df.drop(columns=["leaked_feature"])

print("Leaked feature removed.")
print("Remaining columns:")
print(leak_df.columns.tolist())

Leaked feature removed.
Remaining columns:
['client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'opportunity_proxy']


### Leakage lesson

The leaked feature matched the proxy perfectly with 100% agreement because it was created directly from the proxy label. This is not genuine predictive performance; it is target leakage. I removed the leaked feature before continuing. Future modeling should only use information that would genuinely be available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This warehouse has an unbalanced history, so different clients and content items may not have the same amount of historical data. Some rows may also have GSC data available without GA4 data, so availability needs to be checked before using a signal.

The daily table also contains observations that can be aggregated into overlapping time windows. I therefore need to be careful when creating historical features and future outcomes so that information from the outcome period does not enter the features.

The March 2026 slice is useful for development, but it does not represent every possible client or content history in the warehouse.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema_df = con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print(schema_df["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.